In [3]:
# Cell 1: Imports and dual-platform app identifiers

import os
import json
import time
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from google_play_scraper import app as gp_app

load_dotenv()

# === Dual-platform identifier mapping for 10 selected apps ===
# iOS app_id: numeric ID from App Store URL, e.g.
#   https://apps.apple.com/us/app/tiktok/id835599320 -> 835599320
# Android package_name: from Google Play URL after "?id=", e.g.
#   https://play.google.com/store/apps/details?id=com.zhiliaoapp.musically
#   -> com.zhiliaoapp.musically

APPS = [
    {"name": "TikTok",          "category": "Social Media",     "ios_id": "835599320",   "android_pkg": "com.zhiliaoapp.musically"},
    {"name": "Instagram",       "category": "Social Media",     "ios_id": "389801252",   "android_pkg": "com.instagram.android"},
    {"name": "Airbnb",          "category": "Travel",           "ios_id": "401626263",   "android_pkg": "com.airbnb.android"},
    {"name": "Uber",            "category": "Transportation",   "ios_id": "368677368",   "android_pkg": "com.ubercab"},
    {"name": "DoorDash",        "category": "Food Delivery",    "ios_id": "719972451",   "android_pkg": "com.dd.doordash"},
    {"name": "Amazon Shopping", "category": "E-commerce",       "ios_id": "297606951",   "android_pkg": "com.amazon.mShop.android.shopping"},
    {"name": "PayPal",          "category": "Finance",          "ios_id": "283646709",   "android_pkg": "com.paypal.android.p2pmobile"},
    {"name": "Netflix",         "category": "Entertainment",    "ios_id": "363590051",   "android_pkg": "com.netflix.mediaclient"},
    {"name": "ChatGPT",         "category": "AI Tools",         "ios_id": "6448311069",  "android_pkg": "com.openai.chatgpt"},
    {"name": "Google Maps",     "category": "Navigation",       "ios_id": "585027354",   "android_pkg": "com.google.android.apps.maps"},
]

print(f"Configured {len(APPS)} apps, each with iOS + Android identifiers")
print(f"Categories covered: {sorted(set(a['category'] for a in APPS))}")

Configured 10 apps, each with iOS + Android identifiers
Categories covered: ['AI Tools', 'E-commerce', 'Entertainment', 'Finance', 'Food Delivery', 'Navigation', 'Social Media', 'Transportation', 'Travel']


In [4]:
# Cell 2: Fetch current iOS version data via iTunes Lookup API

def fetch_ios_current(ios_id: str) -> dict | None:
    """
    Fetch current version metadata for one app from the #iTunes Lookup API#.

    Returns a dict with version, release date, release notes, etc.,
    or None if the app cannot be found.
    """
    url = f"https://itunes.apple.com/lookup?id={ios_id}&country=us"
    response = requests.get(url, timeout=10)
    data = response.json()

    if data["resultCount"] == 0:
        return None

    item = data["results"][0]
    return {
        "developer":        item.get("artistName"),
        "store_category":   item.get("primaryGenreName"),
        "version":          item.get("version"),
        "release_date":     item.get("currentVersionReleaseDate"),
        "initial_release":  item.get("releaseDate"),
        "release_notes":    item.get("releaseNotes", ""),
        "bundle_id":        item.get("bundleId"),
        "store_url":        item.get("trackViewUrl"),
    }

# Batch-fetch all apps
ios_records = []
for app_info in APPS:
    print(f"[iOS] Fetching {app_info['name']}...", end=" ")
    try:
        result = fetch_ios_current(app_info["ios_id"])
        if result is None:
            print(f"NOT FOUND (check ios_id: {app_info['ios_id']})")
            continue

        record = {
            "app_name":          app_info["name"],
            "platform":          "iOS",
            "user_category":     app_info["category"],
            **result,
            "is_current":        True,
            "source_url":        f"https://itunes.apple.com/lookup?id={app_info['ios_id']}",
            "data_quality_note": "Current version retrieved from iTunes Lookup API"
        }
        ios_records.append(record)
        print(f"OK  v{result['version']}  ({result['release_date'][:10]})")
    except Exception as e:
        print(f"ERROR: {e}")
    time.sleep(0.5)  # Be polite to Apple's servers

ios_df = pd.DataFrame(ios_records)
print(f"\nDone. Collected {len(ios_df)} iOS current-version rows.")
ios_df.head(3)

[iOS] Fetching TikTok... OK  v45.0.0  (2026-05-06)
[iOS] Fetching Instagram... OK  v428.2.0  (2026-05-05)
[iOS] Fetching Airbnb... OK  v26.19  (2026-05-06)
[iOS] Fetching Uber... OK  v3.720.10000  (2026-05-04)
[iOS] Fetching DoorDash... OK  v8.17.2  (2026-05-05)
[iOS] Fetching Amazon Shopping... OK  v27.9.0  (2026-05-04)
[iOS] Fetching PayPal... OK  v8.106.0  (2026-05-04)
[iOS] Fetching Netflix... OK  v18.30.0  (2026-05-04)
[iOS] Fetching ChatGPT... OK  v1.2026.118  (2026-05-06)
[iOS] Fetching Google Maps... OK  v26.18.0  (2026-05-01)

Done. Collected 10 iOS current-version rows.


,app_name,platform,user_category,developer,store_category,version,release_date,initial_release,release_notes,bundle_id,store_url,is_current,source_url,data_quality_note
0,TikTok,iOS,Social Media,TikTok Ltd.,Entertainment,45.0.0,2026-05-06T11:00:04Z,2014-04-02T22:44:45Z,Squashed bugs for better experience.,com.zhiliaoapp.musically,https://apps.apple.com/us/app/tiktok-videos-sh...,True,https://itunes.apple.com/lookup?id=835599320,Current version retrieved from iTunes Lookup API
1,Instagram,iOS,Social Media,"Instagram, Inc.",Photo & Video,428.2.0,2026-05-05T15:02:20Z,2010-10-06T08:12:41Z,Performance optimizations and stability improv...,com.burbn.instagram,https://apps.apple.com/us/app/instagram/id3898...,True,https://itunes.apple.com/lookup?id=389801252,Current version retrieved from iTunes Lookup API
2,Airbnb,iOS,Travel,"Airbnb, Inc.",Travel,26.19,2026-05-06T21:03:46Z,2010-11-10T20:28:11Z,"Now you can book homes, experiences, and servi...",com.airbnb.app,https://apps.apple.com/us/app/airbnb/id4016262...,True,https://itunes.apple.com/lookup?id=401626263,Current version retrieved from iTunes Lookup API


In [6]:
# Cell 3: Fetch current Android version data via google-play-scraper
#
# Network calls to Google Play occasionally fail with transient errors
# (e.g., "Remote end closed connection without response"). To make data
# collection robust, retries with exponential backoff are built into the
# fetch function itself, not bolted on afterward.

def fetch_android_current(package_name: str) -> dict:
    """Fetch current version metadata for one Android app from Google Play."""
    info = gp_app(package_name, lang='en', country='us')
    return {
        "developer":        info.get("developer"),
        "store_category":   info.get("genre"),
        "version":          info.get("version"),
        "release_date":     info.get("updated"),     # Unix timestamp, converted below
        "initial_release":  None,                    # Not exposed by this library
        "release_notes":    info.get("recentChanges") or "",
        "store_url":        info.get("url"),
        "rating":           info.get("score"),
        "installs":         info.get("installs"),
    }


def fetch_android_with_retry(package_name: str, max_retries: int = 5) -> dict:
    """
    Wrap fetch_android_current with exponential-backoff retry.

    Retries on any exception (transient network errors are most common).
    Waits 1s, 2s, 4s, 8s, 16s between attempts. Re-raises after final failure
    so the caller can decide how to handle a permanently failing package.
    """
    last_error = None
    for attempt in range(max_retries):
        try:
            return fetch_android_current(package_name)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait = 2 ** attempt  # 1, 2, 4, 8, 16 seconds
                print(f"  attempt {attempt+1}/{max_retries} failed "
                      f"({type(e).__name__}); retrying in {wait}s...")
                time.sleep(wait)
    raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# === Batch-fetch all 10 apps ===
android_records = []
permanently_failed = []

for app_info in APPS:
    print(f"[Android] Fetching {app_info['name']}...", end=" ")
    try:
        result = fetch_android_with_retry(app_info["android_pkg"])

        # Convert Unix timestamp to ISO date string
        if result["release_date"]:
            result["release_date"] = datetime.fromtimestamp(
                result["release_date"]
            ).strftime("%Y-%m-%d")

        record = {
            "app_name":          app_info["name"],
            "platform":          "Android",
            "user_category":     app_info["category"],
            **result,
            "is_current":        True,
            "source_url":        f"https://play.google.com/store/apps/details?id={app_info['android_pkg']}",
            "data_quality_note": "Current version retrieved from google-play-scraper"
        }
        android_records.append(record)
        print(f"OK  v{result['version']}  ({result['release_date']})")
    except Exception as e:
        permanently_failed.append({"app": app_info["name"], "error": str(e)})
        print(f"PERMANENTLY FAILED: {type(e).__name__}")

    time.sleep(0.5)  # Be polite to Google's servers between apps

android_df = pd.DataFrame(android_records)

print(f"\nDone. Collected {len(android_df)} / {len(APPS)} Android current-version rows.")
if permanently_failed:
    print(f"Permanently failed apps: {[f['app'] for f in permanently_failed]}")
android_df.head(3)

[Android] Fetching TikTok... OK  vVaries with device  (2026-04-30)
[Android] Fetching Instagram... OK  vVaries with device  (2026-05-04)
[Android] Fetching Airbnb... OK  v26.18  (2026-05-05)
[Android] Fetching Uber... OK  v4.629.10001  (2026-05-01)
[Android] Fetching DoorDash... OK  v15.273.1  (2026-05-04)
[Android] Fetching Amazon Shopping... OK  v32.9.0.100  (2026-04-23)
[Android] Fetching PayPal... OK  v8.105.1  (2026-04-22)
[Android] Fetching Netflix... OK  vVaries with device  (2026-05-04)
[Android] Fetching ChatGPT... OK  v1.2026.118  (2026-05-01)
[Android] Fetching Google Maps... OK  vVaries with device  (2026-04-30)

Done. Collected 10 / 10 Android current-version rows.


,app_name,platform,user_category,developer,store_category,version,release_date,initial_release,release_notes,store_url,rating,installs,is_current,source_url,data_quality_note
0,TikTok,Android,Social Media,TikTok Pte. Ltd.,Social,Varies with device,2026-04-30,None,,https://play.google.com/store/apps/details?id=...,4.001908,"1,000,000,000+",True,https://play.google.com/store/apps/details?id=...,Current version retrieved from google-play-scr...
1,Instagram,Android,Social Media,Instagram,Social,Varies with device,2026-05-04,None,,https://play.google.com/store/apps/details?id=...,4.015670,"5,000,000,000+",True,https://play.google.com/store/apps/details?id=...,Current version retrieved from google-play-scr...
2,Airbnb,Android,Travel,Airbnb,Travel & Local,26.18,2026-05-05,None,,https://play.google.com/store/apps/details?id=...,4.539124,"100,000,000+",True,https://play.google.com/store/apps/details?id=...,Current version retrieved from google-play-scr...


In [7]:
# Cell 4: Merge iOS + Android current versions and persist to CSV

# Standardize column order across both platforms
cols = [
    'app_name', 'platform', 'user_category',
    'developer', 'store_category',
    'version', 'release_date', 'initial_release',
    'is_current', 'release_notes',
    'source_url', 'data_quality_note',
]

current_df = pd.concat([ios_df[cols], android_df[cols]], ignore_index=True)
current_df = current_df.sort_values(['app_name', 'platform']).reset_index(drop=True)

# Save raw snapshot for reproducibility
os.makedirs('data/raw', exist_ok=True)
current_df.to_csv('data/raw/current_versions.csv', index=False, encoding='utf-8-sig')

print(f"Saved {len(current_df)} rows to data/raw/current_versions.csv")
print("\nRow count by app:")
print(current_df.groupby('app_name').size())

Saved 20 rows to data/raw/current_versions.csv

Row count by app:
app_name
Airbnb             2
Amazon Shopping    2
ChatGPT            2
DoorDash           2
Google Maps        2
Instagram          2
Netflix            2
PayPal             2
TikTok             2
Uber               2
dtype: int64


In [13]:
# Cell 5 (revised): Fetch iOS version history via Wayback Machine
#
# Improvements over previous version:
# 1. Increased timeout to 90s (CDX API is slow for popular apps).
# 2. Added retry-with-backoff for the CDX call itself.
# 3. Added a hard cap on snapshots per app to bound total runtime.
# 4. Use HTTPS and gzip — faster transfer, fewer redirects.

WAYBACK_CDX_API = "https://web.archive.org/cdx/search/cdx"  # https not http

REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (research student data collection)",
    "Accept-Encoding": "gzip",
}


def list_wayback_snapshots(
    target_url: str,
    limit_per_year: int = 3,
    max_total: int = 25,
    max_retries: int = 5,
) -> list[str]:
    """
    List Wayback Machine snapshots for a given URL, sampled across time.

    Improvements:
    - Long timeout (CDX is slow for popular pages).
    - Retry on timeout with exponential backoff.
    - Hard cap on total snapshots returned to bound runtime.
    """
    params = {
        "url": target_url,
        "output": "json",
        "fl": "timestamp,statuscode",
        "filter": "statuscode:200",
        "collapse": "timestamp:6",   # one snapshot per month
        "limit": 500,                # Server-side cap; faster response
    }

    last_error = None
    for attempt in range(max_retries):
        try:
            response = requests.get(
                WAYBACK_CDX_API,
                params=params,
                headers=REQUEST_HEADERS,
                timeout=120,          # Generous timeout for slow CDX
            )
            response.raise_for_status()
            break
        except (requests.Timeout, requests.ConnectionError, requests.HTTPError) as e:
            last_error = e
            wait = 2 ** attempt
            print(f"    CDX attempt {attempt+1}/{max_retries} failed ({type(e).__name__}); "
                  f"retrying in {wait}s...")
            time.sleep(wait)
    else:
        print(f"    CDX permanently failed: {last_error}")
        return []

    rows = response.json()
    if len(rows) <= 1:
        return []

    timestamps = [row[0] for row in rows[1:]]

    # Sample evenly across years, capped at limit_per_year per year
    by_year = {}
    for ts in timestamps:
        by_year.setdefault(ts[:4], []).append(ts)

    sampled = []
    for year, ts_list in sorted(by_year.items()):
        step = max(1, len(ts_list) // limit_per_year)
        sampled.extend(ts_list[::step][:limit_per_year])

    return sampled[:max_total]   # Hard cap on total


def fetch_archived_appstore_page(timestamp: str, ios_id: str) -> dict | None:
    """Fetch one archived App Store snapshot and parse out version metadata."""
    archived_url = (
        f"https://web.archive.org/web/{timestamp}/"
        f"https://apps.apple.com/us/app/id{ios_id}"
    )

    try:
        response = requests.get(
            archived_url,
            headers=REQUEST_HEADERS,
            timeout=60,
        )
    except (requests.Timeout, requests.ConnectionError):
        return None

    if response.status_code != 200:
        return None

    html = response.text

    version_match = (
        re.search(r'"softwareVersion"\s*:\s*"([^"]+)"', html)
        or re.search(r'whats-new__latest__version[^>]*>Version\s*([^<\s]+)', html)
        or re.search(r'>\s*Version\s+([0-9][^\s<]+)\s*<', html)
    )
    notes_match = (
        re.search(r'"description"\s*:\s*\{[^}]*"#text"\s*:\s*"([^"]+)"', html)
        or re.search(r'class="whats-new__content"[^>]*>([\s\S]+?)</', html)
        or re.search(r'class="we-truncate__child[^"]*"[^>]*>([\s\S]+?)</p>', html)
    )
    date_match = (
        re.search(r'"datePublished"\s*:\s*"([^"]+)"', html)
        or re.search(r'class="release-date"[^>]*>([^<]+)<', html)
    )

    if not version_match:
        return None

    return {
        "version":      version_match.group(1).strip(),
        "release_date": date_match.group(1).strip() if date_match else None,
        "release_notes": (
            re.sub(r'<[^>]+>', ' ', notes_match.group(1)).strip()
            if notes_match else ""
        ),
        "snapshot_timestamp": timestamp,
    }


# === Batch fetch iOS history via Wayback for all apps ===
ios_history_records = []

for app_info in APPS:
    print(f"\n[Wayback iOS] {app_info['name']}")
    target_url = f"apps.apple.com/us/app/id{app_info['ios_id']}"

    snapshots = list_wayback_snapshots(target_url, limit_per_year=3, max_total=20)
    if not snapshots:
        print("  no snapshots available, skipping")
        continue

    print(f"  found {len(snapshots)} snapshots to sample...")
    seen_versions = set()
    success_count = 0

    for ts in snapshots:
        parsed = fetch_archived_appstore_page(ts, app_info["ios_id"])
        if parsed is None or parsed["version"] in seen_versions:
            continue

        seen_versions.add(parsed["version"])
        success_count += 1

        ios_history_records.append({
            "app_name":          app_info["name"],
            "platform":          "iOS",
            "user_category":     app_info["category"],
            "developer":         None,
            "store_category":    None,
            "version":           parsed["version"],
            "release_date":      parsed["release_date"],
            "initial_release":   None,
            "is_current":        False,
            "release_notes":     parsed["release_notes"],
            "source_url":        f"https://web.archive.org/web/{ts}/https://apps.apple.com/us/app/id{app_info['ios_id']}",
            "data_quality_note": f"Historical snapshot from Wayback Machine ({ts[:8]})",
        })
        time.sleep(0.3)

    print(f"  collected {success_count} unique historical versions")
    time.sleep(1)

ios_history_df = pd.DataFrame(ios_history_records)

# Backfill developer/store_category from current_df
if len(ios_history_df) > 0:
    backfill = current_df[current_df['platform'] == 'iOS'].set_index('app_name')[
        ['developer', 'store_category', 'initial_release']
    ].to_dict('index')

    for col in ['developer', 'store_category', 'initial_release']:
        ios_history_df[col] = ios_history_df['app_name'].map(
            lambda n: backfill.get(n, {}).get(col)
        )

print(f"\n=== Done. Collected {len(ios_history_df)} iOS historical version rows. ===")
ios_history_df.head()



[Wayback iOS] TikTok
    CDX attempt 1/5 failed (HTTPError); retrying in 1s...
    CDX attempt 2/5 failed (HTTPError); retrying in 2s...
  found 15 snapshots to sample...
  collected 14 unique historical versions

[Wayback iOS] Instagram
    CDX attempt 1/5 failed (HTTPError); retrying in 1s...
  found 13 snapshots to sample...
  collected 2 unique historical versions

[Wayback iOS] Airbnb
    CDX attempt 1/5 failed (SSLError); retrying in 1s...
    CDX attempt 2/5 failed (SSLError); retrying in 2s...
    CDX attempt 3/5 failed (SSLError); retrying in 4s...
    CDX attempt 4/5 failed (SSLError); retrying in 8s...
    CDX attempt 5/5 failed (SSLError); retrying in 16s...
    CDX permanently failed: HTTPSConnectionPool(host='web.archive.org', port=443): Max retries exceeded with url: /cdx/search/cdx?url=apps.apple.com%2Fus%2Fapp%2Fid401626263&output=json&fl=timestamp%2Cstatuscode&filter=statuscode%3A200&collapse=timestamp%3A6&limit=500 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTE

,app_name,platform,user_category,developer,store_category,version,release_date,initial_release,is_current,release_notes,source_url,data_quality_note
0,TikTok,iOS,Social Media,TikTok Ltd.,Entertainment,11.6.0,2014-04-02,2014-04-02T22:44:45Z,False,,https://web.archive.org/web/20190613013151/htt...,Historical snapshot from Wayback Machine (2019...
1,TikTok,iOS,Social Media,TikTok Ltd.,Entertainment,12.1.0,2014-04-02,2014-04-02T22:44:45Z,False,,https://web.archive.org/web/20190716084654/htt...,Historical snapshot from Wayback Machine (2019...
2,TikTok,iOS,Social Media,TikTok Ltd.,Entertainment,12.6.1,2014-04-02,2014-04-02T22:44:45Z,False,,https://web.archive.org/web/20190820230840/htt...,Historical snapshot from Wayback Machine (2019...
3,TikTok,iOS,Social Media,TikTok Ltd.,Entertainment,15.3.0,"Apr 1, 2014",2014-04-02T22:44:45Z,False,,https://web.archive.org/web/20200320214910/htt...,Historical snapshot from Wayback Machine (2020...
4,TikTok,iOS,Social Media,TikTok Ltd.,Entertainment,16.0.0,"Apr 1, 2014",2014-04-02T22:44:45Z,False,,https://web.archive.org/web/20200519013448/htt...,Historical snapshot from Wayback Machine (2020...


In [14]:
# Cell 5.5: Document apps where iOS historical data could not be retrieved
#
# Four apps (Airbnb, Uber, Amazon, ChatGPT) returned no historical iOS
# snapshots from the Wayback Machine due to repeated 503/SSL errors on
# the CDX search endpoint. We document this transparently so it shows up
# in the final spreadsheet as a known data gap.

apps_with_ios_history = set(ios_history_df['app_name'].unique()) if len(ios_history_df) else set()
all_app_names = {a['name'] for a in APPS}
ios_history_missing = sorted(all_app_names - apps_with_ios_history)

print("Apps with no iOS historical data (Wayback CDX unavailable):")
for name in ios_history_missing:
    print(f"  - {name}")

# Update the data_quality_note for current iOS rows of these apps
# so the limitation is visible in the final output
mask = (current_df['platform'] == 'iOS') & (current_df['app_name'].isin(ios_history_missing))
current_df.loc[mask, 'data_quality_note'] = (
    "Current version retrieved from iTunes Lookup API. "
    "iOS historical versions unavailable: Wayback Machine CDX returned "
    "503/SSL errors during collection."
)
print(f"\nFlagged {mask.sum()} rows with the unavailability note.")

Apps with no iOS historical data (Wayback CDX unavailable):
  - Airbnb
  - Amazon Shopping
  - ChatGPT
  - Uber

Flagged 4 rows with the unavailability note.


In [19]:
# Cell 6 (revised): Fetch Android version history with stratified sampling
#
# Strategy redesign:
#   1. Pull a LARGE candidate pool from APKMirror (up to 10 pages = ~100 versions per app)
#   2. Immediately apply temporal stratified sampling — keep ~6 versions per app,
#      spread evenly across the full time range
#   3. Return only the sampled rows
#
# This produces a dataset comparable in shape to the iOS history: a modest
# number of versions per app, but spread across a meaningful time window.
# Downstream cells receive enrichment-ready samples instead of clustered raw data.

from bs4 import BeautifulSoup
from urllib.parse import urljoin

APKMIRROR_URLS = {
    "TikTok":          "https://www.apkmirror.com/apk/tiktok-pte-ltd/tik-tok/",
    "Instagram":       "https://www.apkmirror.com/apk/instagram/instagram-instagram/",
    "Airbnb":          "https://www.apkmirror.com/apk/airbnb/airbnb/",
    "Uber":            "https://www.apkmirror.com/apk/uber-technologies-inc/uber/",
    "DoorDash":        "https://www.apkmirror.com/apk/doordash/doordash-food-delivery/",
    "Amazon Shopping": "https://www.apkmirror.com/apk/amazon-mobile-llc/amazon-shopping/",
    "PayPal":          "https://www.apkmirror.com/apk/paypal-mobile/paypal-mobile-payments/",
    "Netflix":         "https://www.apkmirror.com/apk/netflix-inc/netflix/",
    "ChatGPT":         "https://www.apkmirror.com/apk/openai/chatgpt/",
    "Google Maps":     "https://www.apkmirror.com/apk/google-inc/maps/",
}

APKMIRROR_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
}


def fetch_apkmirror_versions(app_url: str, max_pages: int = 10) -> list[dict]:
    """
    Scrape the APKMirror app listing page(s) for version history.

    Returns a list of dicts: version, release_date, version_type, detail_url.
    Stops early if Cloudflare blocks or no rows are found.
    """
    versions = []
    for page_num in range(1, max_pages + 1):
        page_url = app_url if page_num == 1 else f"{app_url}?page={page_num}"
        try:
            response = requests.get(page_url, headers=APKMIRROR_HEADERS, timeout=20)
        except Exception as e:
            print(f"    page {page_num} fetch error: {type(e).__name__}")
            break

        if response.status_code == 403:
            print(f"    page {page_num} returned 403 (Cloudflare block)")
            break
        if response.status_code != 200:
            print(f"    page {page_num} returned {response.status_code}")
            break
        if "challenge-platform" in response.text or "cf-mitigated" in response.headers:
            print(f"    page {page_num} hit Cloudflare challenge")
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        rows = soup.select('div.appRow') or soup.select('div.listWidget div.table-row')
        if not rows:
            rows = soup.select('h5.appRowTitle') or soup.select('a.fontBlack')
        if not rows:
            print(f"    page {page_num}: no version rows found, stopping")
            break

        page_count = 0
        for row in rows:
            link = row.find('a', class_='fontBlack') or row.find('a', href=True)
            if not link:
                continue
            version_text = link.get_text(strip=True)
            detail_url = urljoin(app_url, link.get('href', ''))
            date_tag = row.find('span', class_='dateyear_utc') or row.find(class_='dateyear')
            release_date = date_tag.get_text(strip=True) if date_tag else None
            type_tag = row.find(class_='apkm-badge')
            version_type = type_tag.get_text(strip=True) if type_tag else "Stable"

            versions.append({
                "version_text":  version_text,
                "release_date":  release_date,
                "version_type":  version_type,
                "detail_url":    detail_url,
            })
            page_count += 1

        print(f"    page {page_num}: {page_count} versions")
        time.sleep(2)  # Polite delay between pages

    return versions


def stratified_sample_by_time(versions: list[dict], n_samples: int = 6) -> list[dict]:
    """
    Sample n_samples versions from the candidate pool, evenly spaced across
    the full time range. Always includes the earliest and latest versions
    so the time span is preserved.
    """
    if not versions:
        return []

    # Parse dates into Timestamps for sorting; rows with unparseable dates go last
    parsed = []
    for v in versions:
        try:
            ts = pd.to_datetime(v["release_date"], errors='coerce', utc=True)
            if pd.notna(ts):
                parsed.append((ts, v))
        except Exception:
            continue

    if not parsed:
        return versions[:n_samples]  # Fallback: just take first N

    parsed.sort(key=lambda x: x[0])

    if len(parsed) <= n_samples:
        return [v for _, v in parsed]

    # Pick evenly-spaced indices, always including endpoints
    n = len(parsed)
    indices = [int(round(i * (n - 1) / (n_samples - 1))) for i in range(n_samples)]
    indices = sorted(set(indices))  # Dedupe in case of rounding collisions
    return [parsed[i][1] for i in indices]


# === Batch fetch + sample for all apps ===
android_history_records = []
apkmirror_failed_apps = []

for app_info in APPS:
    name = app_info["name"]
    print(f"\n[APKMirror] {name}")

    if name not in APKMIRROR_URLS:
        print(f"  no APKMirror URL configured, skipping")
        apkmirror_failed_apps.append(name)
        continue

    try:
        # Step 1: pull large candidate pool
        candidate_pool = fetch_apkmirror_versions(APKMIRROR_URLS[name], max_pages=10)
    except Exception as e:
        print(f"  unhandled error: {type(e).__name__}: {e}")
        apkmirror_failed_apps.append(name)
        continue

    if not candidate_pool:
        print(f"  no versions extracted at all")
        apkmirror_failed_apps.append(name)
        continue

    # Step 2: stratified temporal sampling
    sampled = stratified_sample_by_time(candidate_pool, n_samples=6)
    print(f"  pool={len(candidate_pool)}, sampled={len(sampled)}")

    # Step 3: emit records
    for v in sampled:
        m = re.search(r'\b(\d+(?:\.\d+){1,4}[a-zA-Z0-9\-]*)\b', v["version_text"])
        version_num = m.group(1) if m else v["version_text"]

        android_history_records.append({
            "app_name":          name,
            "platform":          "Android",
            "user_category":     app_info["category"],
            "developer":         None,
            "store_category":    None,
            "version":           version_num,
            "release_date":      v["release_date"],
            "initial_release":   None,
            "is_current":        False,
            "release_notes":     "",   # to be filled by Cell 6.3
            "source_url":        v["detail_url"],
            "data_quality_note": (
                f"Historical version from APKMirror "
                f"(stratified sample of {len(candidate_pool)} pool; "
                f"version_type={v['version_type']})"
            ),
        })

    time.sleep(3)  # Polite delay between apps

# Build dataframe and backfill developer
android_history_df = pd.DataFrame(android_history_records)

if len(android_history_df) > 0:
    backfill = current_df[current_df['platform'] == 'Android'].set_index('app_name')[
        ['developer', 'store_category']
    ].to_dict('index')
    for col in ['developer', 'store_category']:
        android_history_df[col] = android_history_df['app_name'].map(
            lambda n: backfill.get(n, {}).get(col)
        )

print(f"\n=== Done. Final Android historical sample: {len(android_history_df)} rows ===")
if apkmirror_failed_apps:
    print(f"Apps with no APKMirror data: {apkmirror_failed_apps}")
android_history_df.head(10)


[APKMirror] TikTok
    page 1: 47 versions
    page 2: 50 versions
    page 3: 47 versions
    page 4: 50 versions
    page 5: 47 versions
    page 6: 50 versions
    page 7: 50 versions
    page 8: 47 versions
    page 9: 47 versions
    page 10: 47 versions
  pool=482, sampled=6

[APKMirror] Instagram
    page 1 returned 403 (Cloudflare block)
  no versions extracted at all

[APKMirror] Airbnb
    page 1 returned 403 (Cloudflare block)
  no versions extracted at all

[APKMirror] Uber
    page 1 returned 403 (Cloudflare block)
  no versions extracted at all

[APKMirror] DoorDash
    page 1 returned 403 (Cloudflare block)
  no versions extracted at all

[APKMirror] Amazon Shopping
    page 1: 47 versions
    page 2: 47 versions
    page 3: 47 versions
    page 4: 47 versions
    page 5: 47 versions
    page 6: 47 versions
    page 7: 47 versions
    page 8: 50 versions
    page 9: 47 versions
    page 10: 47 versions
  pool=473, sampled=6

[APKMirror] PayPal
    page 1 returned 403 (C

,app_name,platform,user_category,developer,store_category,version,release_date,initial_release,is_current,release_notes,source_url,data_quality_note
0,TikTok,Android,Social Media,TikTok Pte. Ltd.,Social,43.9.3,"March 3, 2026",None,False,,https://www.apkmirror.com/apk/tiktok-pte-ltd/t...,Historical version from APKMirror (stratified ...
1,TikTok,Android,Social Media,TikTok Pte. Ltd.,Social,44.4.1,"March 24, 2026",None,False,,https://www.apkmirror.com/apk/tiktok-pte-ltd/t...,Historical version from APKMirror (stratified ...
2,TikTok,Android,Social Media,TikTok Pte. Ltd.,Social,44.4.1,"March 24, 2026",None,False,,https://www.apkmirror.com/apk/tiktok-pte-ltd/t...,Historical version from APKMirror (stratified ...
3,TikTok,Android,Social Media,TikTok Pte. Ltd.,Social,44.2.2,"March 24, 2026",None,False,,https://www.apkmirror.com/apk/tiktok-pte-ltd/t...,Historical version from APKMirror (stratified ...
4,TikTok,Android,Social Media,TikTok Pte. Ltd.,Social,44.5.1,"April 3, 2026",None,False,,https://www.apkmirror.com/apk/tiktok-pte-ltd/t...,Historical version from APKMirror (stratified ...
5,TikTok,Android,Social Media,TikTok Pte. Ltd.,Social,44.6.2,"April 4, 2026",None,False,,https://www.apkmirror.com/apk/tiktok-pte-ltd/t...,Historical version from APKMirror (stratified ...
6,Amazon Shopping,Android,E-commerce,Amazon Mobile LLC,Shopping,32.1.2.100,"January 10, 2026",None,False,,https://www.apkmirror.com/apk/amazon-mobile-ll...,Historical version from APKMirror (stratified ...
7,Amazon Shopping,Android,E-commerce,Amazon Mobile LLC,Shopping,32.3.0.100,"February 3, 2026",None,False,,https://www.apkmirror.com/apk/amazon-mobile-ll...,Historical version from APKMirror (stratified ...
8,Amazon Shopping,Android,E-commerce,Amazon Mobile LLC,Shopping,32.5.0.100,"March 3, 2026",None,False,,https://www.apkmirror.com/apk/amazon-mobile-ll...,Historical version from APKMirror (stratified ...
9,Amazon Shopping,Android,E-commerce,Amazon Mobile LLC,Shopping,32.6.0.100,"March 17, 2026",None,False,,https://www.apkmirror.com/apk/amazon-mobile-ll...,Historical version from APKMirror (stratified ...


In [22]:
# Cell 6.1 (revised): Normalize release_date format across all dataframes
#
# Why this is non-trivial: the four data sources return dates in formats
# with INCONSISTENT timezone metadata.
#   - iTunes API:   "2026-05-06T21:03:46Z"  -> tz-aware UTC
#   - Wayback iOS:  "Apr 1, 2014"           -> tz-naive
#   - APKMirror:    "April 4, 2026"         -> tz-naive
#   - google-play:  "2026-05-04"            -> tz-naive
#
# pandas refuses to compare tz-aware and tz-naive timestamps in the same
# column (TypeError). Since this study only needs day-level granularity,
# we strip timezone info from everything and standardize on tz-naive
# date-only timestamps.

def normalize_date(date_value) -> pd.Timestamp | None:
    """Best-effort date parsing, returning a tz-naive Timestamp at day precision."""
    if pd.isna(date_value) or date_value in (None, "", "None"):
        return None
    try:
        ts = pd.to_datetime(date_value, errors='coerce', utc=True)
        if pd.isna(ts):
            return None
        # Strip timezone, then truncate to date (drop hours/minutes/seconds)
        return ts.tz_convert(None).normalize()
    except Exception:
        return None


for df_name, df in [("current_df", current_df),
                    ("ios_history_df", ios_history_df),
                    ("android_history_df", android_history_df)]:
    if len(df) == 0:
        continue
    before = df['release_date'].notna().sum()
    df['release_date'] = df['release_date'].apply(normalize_date)
    after = df['release_date'].notna().sum()
    print(f"{df_name}: {before} -> {after} valid dates after normalization "
          f"(dtype: {df['release_date'].dtype})")

# Sanity check: print date ranges
print("\nDate range per source:")
for df_name, df in [("current", current_df),
                    ("ios history", ios_history_df),
                    ("android history", android_history_df)]:
    if len(df) and df['release_date'].notna().any():
        d = df['release_date'].dropna()
        print(f"  {df_name}: {d.min().date()} to {d.max().date()} ({len(d)} rows)")

# Also check time span per app — this answers our key question:
# "Did APKMirror only give us recent versions?"
print("\nAndroid history time span per app:")
if len(android_history_df) > 0:
    span = (
        android_history_df.dropna(subset=['release_date'])
        .groupby('app_name')['release_date']
        .agg(['min', 'max', 'count'])
    )
    span['span_days'] = (span['max'] - span['min']).dt.days
    print(span)

current_df: 20 -> 20 valid dates after normalization (dtype: datetime64[ns])
ios_history_df: 40 -> 40 valid dates after normalization (dtype: datetime64[ns])
android_history_df: 30 -> 30 valid dates after normalization (dtype: datetime64[ns])

Date range per source:
  current: 2026-04-22 to 2026-05-06 (20 rows)
  ios history: 2008-07-11 to 2019-02-03 (40 rows)
  android history: 2026-01-10 to 2026-05-06 (30 rows)

Android history time span per app:
                       min        max  count  span_days
app_name                                               
Amazon Shopping 2026-01-10 2026-04-29      6        109
ChatGPT         2026-03-03 2026-05-06      6         64
Google Maps     2026-04-20 2026-04-29      6          9
Netflix         2026-04-22 2026-05-02      6         10
TikTok          2026-03-03 2026-04-04      6         32


In [24]:
# Cell 6.2 (revised): Fetch release notes from APKMirror version detail pages
#
# After Cell 6's stratified sampling, android_history_df contains ~30 rows,
# each with a `source_url` pointing to that version's APKMirror detail page.
# We now fetch the release notes text from each detail page.
#
# Note: variable name changed from android_sampled to android_history_df
# to match the unified design where Cell 6 produces sampled rows directly.

def fetch_apkmirror_release_notes(detail_url: str) -> str | None:
    """Fetch release notes text from a single APKMirror version page."""
    try:
        response = requests.get(detail_url, headers=APKMIRROR_HEADERS, timeout=20)
    except Exception:
        return None

    if response.status_code != 200:
        return None
    if "challenge-platform" in response.text:
        return None

    soup = BeautifulSoup(response.text, 'html.parser')

    # APKMirror's release notes live in different containers depending on
    # the page template — try several selectors for resilience.
    notes_div = (
        soup.find('div', class_='notes')
        or soup.find('div', id='whats-new')
        or soup.find('div', class_='wm')
        or soup.find('div', class_='whatsnew')
    )

    if notes_div:
        text = notes_div.get_text(separator=' ', strip=True)
        text = re.sub(r'\s+', ' ', text).strip()
        return text if text else None

    return None


# === Enrich each row of the (already sampled) android_history_df ===
enriched_count = 0
failed_count = 0
blocked_count = 0

print(f"Enriching {len(android_history_df)} sampled Android rows with release notes...\n")

for idx in android_history_df.index:
    detail_url = android_history_df.at[idx, 'source_url']
    app_name   = android_history_df.at[idx, 'app_name']
    version    = android_history_df.at[idx, 'version']

    print(f"  [{app_name} v{version}]...", end=" ")
    notes = fetch_apkmirror_release_notes(detail_url)

    if notes:
        android_history_df.at[idx, 'release_notes'] = notes
        # Append to existing data_quality_note rather than overwrite
        existing_note = android_history_df.at[idx, 'data_quality_note']
        android_history_df.at[idx, 'data_quality_note'] = (
            existing_note + "; release_notes from APKMirror detail page"
        )
        enriched_count += 1
        print(f"OK  ({len(notes)} chars)")
    else:
        failed_count += 1
        print("blocked or no notes")

    time.sleep(3)  # Cautious: APKMirror is sensitive

print(f"\n=== Enrichment summary ===")
print(f"  Enriched: {enriched_count}")
print(f"  Failed:   {failed_count}")
print(f"  Coverage: {enriched_count / len(android_history_df) * 100:.0f}%")

Enriching 30 sampled Android rows with release notes...

  [TikTok v43.9.3]... OK  (39 chars)
  [TikTok v44.4.1]... OK  (80 chars)
  [TikTok v44.4.1]... OK  (80 chars)
  [TikTok v44.2.2]... OK  (33 chars)
  [TikTok v44.5.1]... OK  (70 chars)
  [TikTok v44.6.2]... OK  (84 chars)
  [Amazon Shopping v32.1.2.100]... OK  (269 chars)
  [Amazon Shopping v32.3.0.100]... OK  (269 chars)
  [Amazon Shopping v32.5.0.100]... OK  (269 chars)
  [Amazon Shopping v32.6.0.100]... OK  (269 chars)
  [Amazon Shopping v32.8.0.100]... OK  (269 chars)
  [Amazon Shopping v32.9.0.100]... OK  (269 chars)
  [Netflix v9.63.0]... OK  (128 chars)
  [Netflix v9.63.1]... OK  (128 chars)
  [Netflix v9.64.0]... OK  (128 chars)
  [Netflix v9.64.0]... OK  (128 chars)
  [Netflix v9.65.0]... OK  (128 chars)
  [Netflix v9.65.0]... OK  (128 chars)
  [ChatGPT v1.2026.055]... OK  (54 chars)
  [ChatGPT v1.2026.069]... OK  (28 chars)
  [ChatGPT v1.2026.083]... OK  (28 chars)
  [ChatGPT v1.2026.090]... OK  (28 chars)
  [ChatGPT v1

In [26]:
# Cell 7: Assemble final dataset
#
# Combine the three dataframes into the final analysis-ready dataset.
# Cell 6 already produced android_history_df with stratified samples,
# so no separate "android_sampled" variable exists.

# Standardize column order across all sources
cols = [
    'app_name', 'platform', 'user_category',
    'developer', 'store_category',
    'version', 'release_date', 'initial_release',
    'is_current', 'release_notes',
    'source_url', 'data_quality_note',
]

# Combine: current versions (20) + iOS history (40) + Android history (30)
final_df = pd.concat([
    current_df[cols],
    ios_history_df[cols],
    android_history_df[cols],
], ignore_index=True)

# Sort for readability
final_df = final_df.sort_values(
    ['app_name', 'platform', 'release_date'],
    na_position='last',
).reset_index(drop=True)

# Persist
os.makedirs('data/raw', exist_ok=True)
final_df.to_csv('data/raw/dataset_pre_llm.csv', index=False, encoding='utf-8-sig')

# Diagnostic
print(f"=== Final dataset assembled: {len(final_df)} rows ===\n")

# How many rows have usable release_notes?
has_notes = final_df['release_notes'].fillna('').str.strip().str.len() > 0
print(f"Rows with release_notes: {has_notes.sum()} / {len(final_df)}")

print(f"\nBreakdown by platform:")
print(final_df.groupby('platform').size())

print(f"\nBreakdown by app and platform:")
print(final_df.groupby(['app_name', 'platform']).size().unstack(fill_value=0))

print(f"\nDate coverage:")
valid_dates = final_df.dropna(subset=['release_date'])
if len(valid_dates):
    print(f"  Earliest: {valid_dates['release_date'].min().date()}")
    print(f"  Latest:   {valid_dates['release_date'].max().date()}")

=== Final dataset assembled: 90 rows ===

Rows with release_notes: 38 / 90

Breakdown by platform:
platform
Android    40
iOS        50
dtype: int64

Breakdown by app and platform:
platform         Android  iOS
app_name                     
Airbnb                 1    1
Amazon Shopping        7    1
ChatGPT                7    1
DoorDash               1    6
Google Maps            7    2
Instagram              1    3
Netflix                7    9
PayPal                 1   11
TikTok                 7   15
Uber                   1    1

Date coverage:
  Earliest: 2008-07-11
  Latest:   2026-05-06


In [27]:
# Cell 7.5 (optional): Symmetrize iOS history sampling
#
# After observing that iOS history has uneven counts per app (TikTok 14
# vs Google Maps 1), we re-apply temporal stratified sampling on iOS so
# both platforms have ~6 rows per app — enabling cross-platform comparison.
#
# Run this only if you want symmetric coverage; skip if you prefer to
# preserve all iOS data points.

def stratified_sample_dataframe(df: pd.DataFrame, group_col: str, n_per_group: int = 6) -> pd.DataFrame:
    """Stratified sample within each group based on release_date."""
    out = []
    for name, g in df.groupby(group_col):
        valid = g.dropna(subset=['release_date']).sort_values('release_date')
        if len(valid) <= n_per_group:
            out.append(valid)
            continue
        n = len(valid)
        idx = sorted(set(int(round(i * (n - 1) / (n_per_group - 1))) for i in range(n_per_group)))
        out.append(valid.iloc[idx])
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


ios_history_df = stratified_sample_dataframe(ios_history_df, 'app_name', n_per_group=6)
print(f"iOS history reduced to {len(ios_history_df)} rows after symmetric sampling")
print(ios_history_df.groupby('app_name').size())

iOS history reduced to 26 rows after symmetric sampling
app_name
DoorDash       5
Google Maps    1
Instagram      2
Netflix        6
PayPal         6
TikTok         6
dtype: int64


In [28]:
# Cell 7.6: Backfill release_notes for current Android versions
#
# google-play-scraper returned empty release_notes for our 10 Android current
# versions because the library's `recentChanges` field doesn't always populate.
# We can scrape the same data directly from the Google Play web page, which
# does show "What's new" for most apps.
#
# This is the single highest-ROI enrichment step we can do: 10 rows that are
# trivially available via direct HTTP, no Cloudflare on Google Play.

def fetch_googleplay_whatsnew(package_name: str) -> str | None:
    """Scrape 'What's new' text directly from a Google Play store page."""
    url = f"https://play.google.com/store/apps/details?id={package_name}&hl=en"
    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0"},
            timeout=15,
        )
    except Exception:
        return None

    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, 'html.parser')

    # Google Play's "What's new" is in a div with itemprop or a section
    # Try multiple selectors because Google's HTML changes frequently
    candidates = [
        soup.find('div', {'itemprop': 'description'}),
        soup.find('section', {'aria-label': lambda x: x and "what's new" in x.lower()}),
    ]
    # Generic fallback: find any div that looks like changelog content
    for div in soup.find_all('div'):
        text = div.get_text(separator=' ', strip=True)
        if "What's new" in text and len(text) < 2000:
            candidates.append(div)
            break

    for c in candidates:
        if c is None:
            continue
        text = c.get_text(separator=' ', strip=True)
        text = re.sub(r'\s+', ' ', text)
        # Strip the "What's new" header itself
        text = re.sub(r"^What['']s new\s*", '', text, flags=re.IGNORECASE).strip()
        if 10 < len(text) < 2000:
            return text

    return None


# Find Android current rows missing release_notes in final_df
mask_target = (
    (final_df['platform'] == 'Android')
    & (final_df['is_current'] == True)
    & (final_df['release_notes'].fillna('').str.strip().str.len() == 0)
)
target_indices = final_df[mask_target].index.tolist()

print(f"Backfilling release_notes for {len(target_indices)} Android current rows...\n")

# Build a name -> package_name map from APPS list
pkg_map = {a['name']: a['android_pkg'] for a in APPS}

for idx in target_indices:
    app_name = final_df.at[idx, 'app_name']
    pkg = pkg_map.get(app_name)
    if not pkg:
        continue

    print(f"  [{app_name}]...", end=" ")
    notes = fetch_googleplay_whatsnew(pkg)
    if notes:
        final_df.at[idx, 'release_notes'] = notes
        existing = final_df.at[idx, 'data_quality_note'] or ""
        final_df.at[idx, 'data_quality_note'] = (
            existing + "; release_notes backfilled from Google Play web page"
        )
        print(f"OK ({len(notes)} chars)")
    else:
        print("not found")

    time.sleep(2)

# Re-save
final_df.to_csv('data/raw/dataset_pre_llm.csv', index=False, encoding='utf-8-sig')

# Show updated coverage
has_notes = final_df['release_notes'].fillna('').str.strip().str.len() > 0
print(f"\nUpdated coverage: {has_notes.sum()} / {len(final_df)} rows have release_notes")

Backfilling release_notes for 10 Android current rows...

  [Airbnb]... not found
  [Amazon Shopping]... OK (269 chars)
  [ChatGPT]... not found
  [DoorDash]... OK (491 chars)
  [Google Maps]... OK (181 chars)
  [Instagram]... not found
  [Netflix]... OK (128 chars)
  [PayPal]... OK (127 chars)
  [TikTok]... OK (36 chars)
  [Uber]... OK (300 chars)

Updated coverage: 45 / 90 rows have release_notes
